[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/00_setup/environment_setup.ipynb)

# 00 . Environment Setup

## Concept — Environment Setup (revision notes)

**What it is**
- The first-run checklist for any PyTorch project: which versions you have, which accelerator (CUDA GPU, Apple MPS, or plain CPU) is available, and how to make runs repeatable.
- Everything in this repo runs on CPU; if a GPU is present the same code uses it via one `device` variable.

**Why it matters**
- "It works on my machine" bugs are almost always version, device, or randomness differences.
- A one-line device switch lets the same notebook run on a laptop, Colab, or a GPU server.
- Seeding is the difference between a result you can debug and one that changes every run.

**When to use**
- At the top of every notebook or script: detect device, seed, print versions.
- Again when a result is not reproducible or a CUDA / MPS error appears.

**Math & intuition**
- Random number generators are deterministic sequences from a starting state (the seed). Same seed + same ops + same hardware/library = same numbers.
- Python, NumPy and torch each keep their OWN generator, so seeding one does not seed the others.
- GPU speed comes from massive parallelism: it wins for large matmuls (cost ~ O(n^3) flops), but tiny tensors are dominated by launch overhead, so CPU can be faster for small work.
- GPU kernels run asynchronously, so timing needs `torch.cuda.synchronize()` before reading the clock.

### 1. Python, OS and library versions

**What this cell does (plain language):** we print the Python, OS and PyTorch versions and whether this torch build was compiled with CUDA support. Paste this output into any bug report: it answers the first questions people ask.

In [1]:
import sys, platform
import torch

print("python   :", sys.version.split()[0])
print("platform :", platform.platform())
print("torch    :", torch.__version__)
print("built with CUDA :", torch.version.cuda)      # None for CPU-only wheels
print("cuDNN version   :", torch.backends.cudnn.version())  # None if unavailable
print("CPU threads torch will use:", torch.get_num_threads())

python   : 3.11.15
platform : Linux-6.18.44-fc-v70-x86_64-with-glibc2.39
torch    : 2.14.1+cu130
built with CUDA : 13.0
cuDNN version   : 92400
CPU threads torch will use: 4


### 2. Device detection (CUDA / MPS / CPU)

**What this cell does (plain language):** we ask torch which accelerators exist and pick the best one in priority order: CUDA (NVIDIA), then MPS (Apple silicon), then CPU. We wrap the logic in a function so every later notebook can reuse it, and print details for whichever device we got.

In [2]:
def get_device(prefer=None):
    """Return the best available torch.device. `prefer` forces e.g. 'cpu'."""
    if prefer is not None:
        return torch.device(prefer)
    if torch.cuda.is_available():
        return torch.device("cuda")
    # MPS = Apple Metal backend; guard with getattr for older torch builds
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print("chosen device:", device)
print("cuda available:", torch.cuda.is_available())
print("mps  available:", getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available())

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print("GPU:", props.name, "| memory GB:", round(props.total_memory / 1e9, 1),
          "| compute capability:", f"{props.major}.{props.minor}")
    print("GPU count:", torch.cuda.device_count())
else:
    print("No GPU detected -> everything in this repo still runs on CPU.")

chosen device: cpu
cuda available: False
mps  available: False
No GPU detected -> everything in this repo still runs on CPU.


### 3. Seeding every random number generator

**What this cell does (plain language):** we write a `seed_everything` helper that seeds Python's `random`, NumPy, and torch (CPU and all CUDA devices). Then we prove it works: reseeding and drawing again gives identical numbers, while forgetting to reseed gives different ones.

In [3]:
import random
import numpy as np

def seed_everything(seed=42):
    random.seed(seed)                  # Python's own RNG
    np.random.seed(seed)               # NumPy's global RNG
    torch.manual_seed(seed)            # torch CPU RNG (also seeds CUDA lazily)
    torch.cuda.manual_seed_all(seed)   # explicit; no-op without CUDA
    return seed

seed_everything(42)
a = torch.rand(3)
seed_everything(42)
b = torch.rand(3)
c = torch.rand(3)                      # NOT reseeded -> continues the sequence

print("a:", a)
print("b:", b)
print("a == b (same seed)       :", torch.equal(a, b))
print("b == c (no reseed between):", torch.equal(b, c))

# torch and numpy are independent streams: seeding one does not seed the other
torch.manual_seed(0); x1 = np.random.rand(); torch.manual_seed(0); x2 = np.random.rand()
print("numpy draws differ after torch seeding only:", x1 != x2)

a: tensor([0.8823, 0.9150, 0.3829])
b: tensor([0.8823, 0.9150, 0.3829])
a == b (same seed)       : True
b == c (no reseed between): False
numpy draws differ after torch seeding only: True


### 4. Sanity tensor operations

**What this cell does (plain language):** we create tensors and run the handful of operations that every model is built from (matmul, broadcasting, reductions, indexing) and check the results against known values. If something here fails, the installation is broken, not your model.

In [4]:
seed_everything(0)
x = torch.arange(6, dtype=torch.float32).reshape(2, 3)   # [[0,1,2],[3,4,5]]
w = torch.ones(3, 2)

y = x @ w                                  # matmul: (2,3) @ (3,2) -> (2,2)
print("x @ w =\n", y)
assert torch.equal(y, torch.tensor([[3., 3.], [12., 12.]]))

bias = torch.tensor([10., 20.])
print("broadcast add:\n", y + bias)      # (2,2) + (2,) -> bias added to each row

print("sum:", x.sum().item(), "| mean:", x.mean().item(), "| row max:", x.max(dim=1).values)
print("softmax rows sum to 1:", torch.softmax(x, dim=1).sum(dim=1))
print("indexing x[:, 1]:", x[:, 1], "| shape/dtype:", tuple(x.shape), x.dtype)
print("all sanity checks passed")

x @ w =
 tensor([[ 3.,  3.],
        [12., 12.]])
broadcast add:
 tensor([[13., 23.],
        [22., 32.]])
sum: 15.0 | mean: 2.5 | row max: tensor([2., 5.])
softmax rows sum to 1: tensor([1., 1.])
indexing x[:, 1]: tensor([1., 4.]) | shape/dtype: (2, 3) torch.float32
all sanity checks passed


### 5. Moving data to a device and common dtypes

**What this cell does (plain language):** we move tensors and a small model to the chosen device with `.to(device)`, look at where they live, and see that mixing devices raises an error. We also peek at dtype sizes, because float32 vs float16 vs bfloat16 matters for memory and speed.

In [5]:
model = torch.nn.Linear(4, 2).to(device)      # moves parameters in place (Modules)
x = torch.randn(5, 4).to(device)              # tensors return a NEW tensor, must reassign
print("param device:", next(model.parameters()).device, "| input device:", x.device)
print("output shape:", tuple(model(x).shape))

# Mixing devices is an error. Only demonstrable when a non-CPU device exists.
if device.type != "cpu":
    try:
        model(torch.randn(5, 4))              # CPU tensor into accelerator model
    except RuntimeError as e:
        print("device mismatch caught:", str(e)[:80], "...")
else:
    print("CPU-only machine: device mismatch can't be shown here, but it is the #1 GPU bug.")

for dt in [torch.float32, torch.float16, torch.bfloat16, torch.float64, torch.int64]:
    print(f"{str(dt):15s} bytes/element = {torch.empty(0, dtype=dt).element_size()}")

param device: cpu | input device: cpu
output shape: (5, 2)
CPU-only machine: device mismatch can't be shown here, but it is the #1 GPU bug.
torch.float32   bytes/element = 4
torch.float16   bytes/element = 2
torch.bfloat16  bytes/element = 2
torch.float64   bytes/element = 8
torch.int64     bytes/element = 8


### 6. Autograd sanity check

**What this cell does (plain language):** we compute a tiny gradient by hand and compare it to what autograd returns. For f(w) = sum((w*x)^2) the derivative is 2*w*x^2, so we know the right answer exactly.

In [6]:
w = torch.tensor([1.0, 2.0, 3.0], requires_grad=True, device=device)
x = torch.tensor([0.5, 0.5, 0.5], device=device)

f = ((w * x) ** 2).sum()
f.backward()

expected = 2 * w.detach() * x ** 2
print("autograd grad :", w.grad.cpu())
print("hand-derived  :", expected.cpu())
print("match:", torch.allclose(w.grad, expected))

autograd grad : tensor([0.5000, 1.0000, 1.5000])
hand-derived  : tensor([0.5000, 1.0000, 1.5000])
match: True


### 7. Benchmarking matmul on the available devices

**What this cell does (plain language):** we time a matrix multiply on CPU (and on the accelerator if there is one). Two details matter for honest numbers: do a warm-up run first (one-time setup costs), and call `synchronize()` before stopping the clock because GPU kernels run asynchronously. We report the best of several trials because a single timing on a shared machine is noisy; absolute times depend on your hardware, so compare relative sizes.

In [7]:
import time

def sync(dev):
    if dev.type == "cuda": torch.cuda.synchronize()
    elif dev.type == "mps": torch.mps.synchronize()

def bench_matmul(n, dev, reps=10, trials=5):
    a = torch.randn(n, n, device=dev); b = torch.randn(n, n, device=dev)
    _ = a @ b; sync(dev)                      # warm-up
    best = float("inf")
    for _ in range(trials):                   # best-of-N filters out noisy neighbours
        t0 = time.perf_counter()
        for _ in range(reps):
            _ = a @ b
        sync(dev)                             # wait for queued kernels to finish
        best = min(best, (time.perf_counter() - t0) / reps)
    return best

devices = [torch.device("cpu")] + ([device] if device.type != "cpu" else [])
print(f"{'n':>6} " + " ".join(f"{str(d):>12}" for d in devices) + "   (ms per matmul)")
for n in [64, 256, 512]:
    times = [bench_matmul(n, d) * 1e3 for d in devices]
    print(f"{n:>6} " + " ".join(f"{t:>12.3f}" for t in times))

# approximate throughput for the largest size on CPU: a matmul costs 2*n^3 flops
n = 512; t = bench_matmul(n, torch.device("cpu"))
print(f"CPU throughput at n={n}: ~{2 * n**3 / t / 1e9:.1f} GFLOP/s")

     n          cpu   (ms per matmul)


    64       22.786


   256       23.289


   512       25.220


CPU throughput at n=512: ~9.7 GFLOP/s


### 8. CPU threads and reproducibility switches

**What this cell does (plain language):** we look at the knobs that affect speed on CPU (thread count) and the switches that trade speed for exact reproducibility (deterministic algorithms, cuDNN benchmark mode). We toggle them and show a deterministic run is repeatable.

In [8]:
print("intra-op threads:", torch.get_num_threads())
print("inter-op threads:", torch.get_num_interop_threads())
# torch.set_num_threads(2)   # uncomment to limit CPU parallelism (e.g. on shared machines)

# cuDNN options only matter on GPU, but are safe to set everywhere:
torch.backends.cudnn.benchmark = False       # True = faster autotuning, but nondeterministic choice
torch.backends.cudnn.deterministic = True    # force deterministic conv algorithms

# ask torch to raise an error if an op has no deterministic implementation
torch.use_deterministic_algorithms(True, warn_only=True)
print("deterministic algorithms on:", torch.are_deterministic_algorithms_enabled())

def tiny_run():
    seed_everything(123)
    m = torch.nn.Sequential(torch.nn.Linear(8, 16), torch.nn.ReLU(), torch.nn.Linear(16, 1))
    xx, yy = torch.randn(32, 8), torch.randn(32, 1)
    opt = torch.optim.SGD(m.parameters(), lr=0.1)
    for _ in range(20):
        opt.zero_grad(); loss = torch.nn.functional.mse_loss(m(xx), yy); loss.backward(); opt.step()
    return loss.item()

r1, r2 = tiny_run(), tiny_run()
print("run 1 loss:", r1, "| run 2 loss:", r2, "| identical:", r1 == r2)
torch.use_deterministic_algorithms(False)    # restore default

intra-op threads: 4
inter-op threads: 4


deterministic algorithms on: True
run 1 loss: 0.6572091579437256 | run 2 loss: 0.6572091579437256 | identical: True


### 9. Checking optional libraries

**What this cell does (plain language):** we check which optional libraries from the wider ecosystem are installed (torchvision, transformers, etc.) and print the install command for any that are missing. Nothing here is required: the notebook finishes either way.

In [9]:
import importlib

optional = {
    "numpy": "numpy", "matplotlib": "matplotlib", "pandas": "pandas",
    "torchvision": "torchvision", "transformers": "transformers",
    "datasets": "datasets", "peft": "peft", "onnx": "onnx",
}
missing = []
for mod, pip_name in optional.items():
    try:
        m = importlib.import_module(mod)
        print(f"[ok]      {mod:13s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"[missing] {mod:13s} -> pip install {pip_name}")
        missing.append(pip_name)
print("\nOne-liner for everything missing:", ("pip install " + " ".join(missing)) if missing else "nothing to install")

[ok]      numpy         2.4.6


[ok]      matplotlib    3.11.2


[ok]      pandas        3.0.6
[missing] torchvision   -> pip install torchvision
[missing] transformers  -> pip install transformers
[missing] datasets      -> pip install datasets
[missing] peft          -> pip install peft
[missing] onnx          -> pip install onnx

One-liner for everything missing: pip install torchvision transformers datasets peft onnx


### 10. Memory footprint: how big is my model?

**What this cell does (plain language):** we count parameters and bytes for a small model, and show how to read GPU memory counters when a CUDA device exists. Knowing parameters x bytes-per-parameter gives a quick lower bound on memory (training needs several times more for gradients and optimizer state).

In [10]:
net = torch.nn.Sequential(torch.nn.Linear(1000, 500), torch.nn.ReLU(), torch.nn.Linear(500, 10))
n_params = sum(p.numel() for p in net.parameters())
n_bytes = sum(p.numel() * p.element_size() for p in net.parameters())
print(f"parameters: {n_params:,}  |  weights in float32: {n_bytes / 1e6:.2f} MB")
print(f"Adam training state is roughly 4x weights (weights + grads + 2 moments): ~{4 * n_bytes / 1e6:.1f} MB")

if device.type == "cuda":
    net = net.to(device)
    print("allocated:", torch.cuda.memory_allocated() / 1e6, "MB | reserved:", torch.cuda.memory_reserved() / 1e6, "MB")
else:
    print("(GPU memory counters need CUDA; on CPU use the arithmetic above or `psutil` for process memory.)")

parameters: 505,510  |  weights in float32: 2.02 MB
Adam training state is roughly 4x weights (weights + grads + 2 moments): ~8.1 MB
(GPU memory counters need CUDA; on CPU use the arithmetic above or `psutil` for process memory.)


## Common bugs

- **Forgetting `.to(device)` on the data (or the model)** -> `RuntimeError: Expected all tensors to be on the same device`. Fix: move both model and every batch to the same device.
- **`x.to(device)` without reassigning** -> the tensor is unchanged (tensors return a copy; only `nn.Module.to` works in place). Fix: `x = x.to(device)`.
- **Seeding only torch** while shuffling with `random`/NumPy -> runs still differ. Fix: use a `seed_everything` helper and a seeded `DataLoader` generator.
- **Timing GPU code without `synchronize()`** -> you only measure kernel launch time and get absurdly fast numbers. Fix: synchronize before reading the clock, after a warm-up.
- **Benchmarking tiny tensors on GPU** and concluding the GPU is slow -> launch overhead dominates. Fix: benchmark realistic sizes.
- **Assuming `cuda.is_available()` is True after `pip install torch`** -> you may have a CPU-only wheel (`torch.version.cuda is None`). Fix: reinstall the CUDA build matching your driver.
- **Expecting bit-identical results across different hardware or torch versions** -> seeds only guarantee reproducibility on the same stack. Fix: record versions and expect tiny numeric drift.
- **Leaving `cudnn.benchmark=True` and expecting determinism** -> algorithm choice can vary between runs. Fix: set `benchmark=False, deterministic=True` when you need repeatability.

## Exercises

**Exercise 1 - A device-agnostic tensor factory.**
Write `make_batch(n, d, device=None)` that returns a random `(n, d)` tensor on the best available device when `device` is None, and test it.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
def make_batch(n, d, device=None):
    dev = device or get_device()
    return torch.randn(n, d, device=dev)

b = make_batch(4, 3)
print(b.shape, b.device)

torch.Size([4, 3]) cpu


**Exercise 2 - Prove seeding scope.**
Show that `torch.manual_seed` does NOT control Python's `random` module: seed torch twice and compare `random.random()` draws.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
torch.manual_seed(0); r1 = random.random()
torch.manual_seed(0); r2 = random.random()
print(r1, r2, "equal?", r1 == r2)   # almost surely different: python RNG untouched

0.052363598850944326 0.08718667752263232 equal? False


**Exercise 3 - Count parameters by hand.**
Build `nn.Linear(10, 5)` and verify that its parameter count equals `10*5 + 5` using `numel()`.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
lin = torch.nn.Linear(10, 5)
total = sum(p.numel() for p in lin.parameters())
print(total, total == 10 * 5 + 5)

55 True


**Exercise 4 - Time matmul vs elementwise.**
Time a 512x512 matmul and a 512x512 elementwise multiply on CPU. Which is slower and by roughly how much? (Matmul does O(n^3) work, elementwise O(n^2).)

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
a = torch.randn(512, 512); b = torch.randn(512, 512)
def timeit(fn, reps=20):
    fn(); t0 = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t0) / reps * 1e3
print(f"matmul: {timeit(lambda: a @ b):.3f} ms | elementwise: {timeit(lambda: a * b):.3f} ms")

matmul: 33.794 ms | elementwise: 37.350 ms


**Exercise 5 - Limit CPU threads.**
Read the thread count, set it to 1, time a matmul, then restore it. Does the time change?

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
orig = torch.get_num_threads()
a = torch.randn(512, 512)
def t(): 
    a @ a; t0 = time.perf_counter()
    for _ in range(10): a @ a
    return (time.perf_counter() - t0) / 10 * 1e3
print("threads", orig, f"-> {t():.3f} ms")
torch.set_num_threads(1); print("threads 1", f"-> {t():.3f} ms")
torch.set_num_threads(orig)

threads 4 -> 32.122 ms
threads 1 -> 10.721 ms


**Exercise 6 - Detect a CPU-only wheel.**
Write a function that returns a human-readable verdict: 'CUDA build + GPU', 'CUDA build but no GPU visible', or 'CPU-only build'.

In [21]:
# Your attempt for Exercise 6 here

**Solution 6**

In [22]:
# Solution 6
def verdict():
    if torch.version.cuda is None: return "CPU-only build"
    return "CUDA build + GPU" if torch.cuda.is_available() else "CUDA build but no GPU visible"
print(verdict())

CUDA build but no GPU visible
